![](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)

# Cap 01 — Estado y Grafos

profesor [Carlos Daniel Jiménez](danieljimenez88m@gmail.com)


Aprenderemos los bloques fundamentales de LangGraph:
- `TypedDict` como estructura de estado
- `StateGraph` para definir el grafo
- `START` y `END` como puntos de entrada/salida
- Nodos como funciones puras
- Visualización con Mermaid

**Dominio**: Películas de Christopher Nolan  
**Flujo**: `START → node_enriquecedor → node_formateador → END`

In [ ]:
# Instalación (ejecutar solo si es necesario)
# !pip install langgraph langchain-core rich

In [ ]:
from typing import TypedDict, Optional, List
from langgraph.graph import StateGraph, START, END
from rich import print as rprint
import json
from pathlib import Path

print("LangGraph importado correctamente ✓")

## 1. Estado con TypedDict

El estado en LangGraph es un `TypedDict` — un diccionario con tipos definidos.  
Cada nodo recibe el estado actual y devuelve un diccionario con los campos que quiere actualizar.

In [ ]:
class NolanState(TypedDict):
    """Estado para el grafo de películas de Nolan."""
    titulo: str
    año: Optional[int]
    temas: List[str]
    resumen_formateado: str
    procesado: bool

# Creamos un estado inicial
estado_inicial = NolanState(
    titulo="Inception",
    año=2010,
    temas=["sueños", "realidad", "culpa"],
    resumen_formateado="",
    procesado=False,
)

rprint(estado_inicial)

## 2. Cargar los datos de Nolan

In [ ]:
DATA_PATH = Path("../00_datos/nolan_films.json")
with open(DATA_PATH, encoding="utf-8") as f:
    nolan_films = json.load(f)

print(f"Cargadas {len(nolan_films)} películas de Nolan")
print(f"Primera película: {nolan_films[0]['titulo']} ({nolan_films[0]['año']})")

## 3. Nodos como funciones puras

Un **nodo** en LangGraph es simplemente una función Python que:
- Recibe el estado actual (diccionario completo)
- Devuelve un diccionario con los campos a actualizar

No necesita devolver el estado completo — solo los campos que cambian.

In [ ]:
def node_enriquecedor(state: NolanState) -> dict:
    """
    Enriquece el estado con datos adicionales del dataset.
    Recibe: titulo, año, temas
    Devuelve: temas actualizados desde el dataset
    """
    titulo = state["titulo"]
    
    # Buscar en el dataset
    pelicula = next(
        (f for f in nolan_films if titulo.lower() in f["titulo"].lower()),
        None
    )
    
    if pelicula:
        return {
            "temas": pelicula["temas"],
            "año": pelicula["año"],
        }
    return {}  # Sin cambios si no se encuentra

def node_formateador(state: NolanState) -> dict:
    """
    Formatea el estado en un texto legible.
    Recibe: titulo, año, temas
    Devuelve: resumen_formateado, procesado=True
    """
    temas_str = ", ".join(state["temas"])
    resumen = (
        f"📽️ **{state['titulo']}** ({state['año']})\n"
        f"Temas principales: {temas_str}"
    )
    return {
        "resumen_formateado": resumen,
        "procesado": True,
    }

print("Nodos definidos ✓")

## 4. Construir y Compilar el Grafo

In [ ]:
# Crear el constructor del grafo
builder = StateGraph(NolanState)

# Añadir nodos
builder.add_node("enriquecedor", node_enriquecedor)
builder.add_node("formateador", node_formateador)

# Añadir aristas (edges)
builder.add_edge(START, "enriquecedor")
builder.add_edge("enriquecedor", "formateador")
builder.add_edge("formateador", END)

# Compilar el grafo
graph = builder.compile()

print("Grafo compilado ✓")

## 5. Visualizar el Grafo

In [ ]:
# Visualización como texto (siempre disponible)
try:
    from IPython.display import Image, display
    png_data = graph.get_graph().draw_mermaid_png()
    display(Image(png_data))
except Exception:
    # Fallback: texto Mermaid
    print(graph.get_graph().draw_mermaid())

## 6. Ejecutar el Grafo con `.invoke()`

In [ ]:
resultado = graph.invoke({
    "titulo": "Inception",
    "año": None,
    "temas": [],
    "resumen_formateado": "",
    "procesado": False,
})

rprint("\n=== Resultado Final ===")
rprint(resultado)

## 7. Trazar la Evolución del Estado

Podemos ver cómo cambia el estado en cada nodo usando `stream()` con `stream_mode="updates"`.

In [ ]:
print("=== Evolución del Estado ===\n")
for step, chunk in enumerate(graph.stream(
    {"titulo": "Memento", "año": None, "temas": [], "resumen_formateado": "", "procesado": False},
    stream_mode="updates"
)):
    print(f"Paso {step + 1}:")
    rprint(chunk)
    print()

## 8. Avanzado: Node de Resumen y Múltiples Películas

In [ ]:
def node_resumen_comparativo(state: NolanState) -> dict:
    """Añade un resumen comparativo con otras películas del mismo año."""
    año = state.get("año")
    misma_epoca = [
        f["titulo"] for f in nolan_films 
        if f.get("año") and abs(f["año"] - (año or 0)) <= 3
        and f["titulo"] != state["titulo"]
    ]
    
    conexiones = ", ".join(misma_epoca[:2]) if misma_epoca else "ninguna registrada"
    resumen_actual = state.get("resumen_formateado", "")
    return {
        "resumen_formateado": resumen_actual + f"\nPelículas de época similar: {conexiones}"
    }

# Nuevo grafo con 3 nodos
builder2 = StateGraph(NolanState)
builder2.add_node("enriquecedor", node_enriquecedor)
builder2.add_node("formateador", node_formateador)
builder2.add_node("resumen_comparativo", node_resumen_comparativo)

builder2.add_edge(START, "enriquecedor")
builder2.add_edge("enriquecedor", "formateador")
builder2.add_edge("formateador", "resumen_comparativo")
builder2.add_edge("resumen_comparativo", END)

graph2 = builder2.compile()

resultado2 = graph2.invoke({
    "titulo": "The Dark Knight",
    "año": None, "temas": [], "resumen_formateado": "", "procesado": False
})
print(resultado2["resumen_formateado"])

## Resumen del Capítulo

| Concepto | Descripción |
|---------|-------------|
| `TypedDict` | Define la estructura del estado |
| `StateGraph` | Constructor del grafo |
| `START`, `END` | Puntos de entrada y salida |
| Nodo | Función pura: `(state) → dict` |
| `.add_node()` | Registra un nodo |
| `.add_edge()` | Conecta nodos |
| `.compile()` | Genera el grafo ejecutable |
| `.invoke()` | Ejecuta y devuelve estado final |
| `.stream()` | Ejecuta devolviendo actualizaciones incrementales |

**Próximo capítulo**: Mensajes y LLMs con `MessagesState`